# Proyecto Avance: EDA Mercado Inmobiliario (Airbnb)

Este cuaderno realiza un analisis exploratorio de datos (EDA) sobre el dataset de Airbnb Price Prediction.

Objetivos:
- Cargar y revisar la calidad de los datos.
- Calcular estadisticas descriptivas.
- Visualizar distribuciones y relaciones con el precio.
- Obtener hallazgos iniciales para el modelado predictivo.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RUTA_DATOS = Path("Datos")
RUTA_FIGS = Path("Visualizaciones")
RUTA_FIGS.mkdir(parents=True, exist_ok=True)

## 1. Carga de datos

Coloca el archivo CSV descargado de Kaggle dentro de `Datos/`.
Este bloque busca automaticamente un `.csv` y lo carga.

In [ ]:
csvs = sorted(RUTA_DATOS.glob("*.csv"))
if not csvs:
    raise FileNotFoundError(
        "No se encontro ningun CSV en Datos/. Descarga el dataset de Kaggle y coloca el archivo ahi."
    )

archivo = csvs[0]
df = pd.read_csv(archivo)

print(f"Archivo cargado: {archivo.name}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df.head())

## 2. Revision general del dataset

In [ ]:
print("Tipos de datos:")
display(df.dtypes.to_frame("dtype"))

print("Valores nulos por columna:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("nulos"))

print("Estadisticas descriptivas (numericas):")
display(df.describe().T)

## 3. Preparacion rapida para visualizacion

In [ ]:
df_eda = df.copy()

# Estandariza columnas
df_eda.columns = (
    df_eda.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Imputacion simple para columnas numericas
for c in df_eda.select_dtypes(include="number").columns:
    df_eda[c] = df_eda[c].fillna(df_eda[c].median())

# Imputacion simple para columnas categoricas
for c in df_eda.select_dtypes(exclude="number").columns:
    moda = df_eda[c].mode(dropna=True)
    if len(moda) > 0:
        df_eda[c] = df_eda[c].fillna(moda[0])

print(f"Nulos restantes: {int(df_eda.isnull().sum().sum())}")
display(df_eda.head())

## 4. Visualizaciones

Se generan y guardan:
- Histograma de variable numerica (precio)
- Box plot de precio
- Scatter de precio vs una variable numerica relevante
- Heatmap de correlaciones

In [ ]:
num_cols = df_eda.select_dtypes(include="number").columns.tolist()
if not num_cols:
    raise ValueError("El dataset no tiene columnas numericas para graficar.")

# Detecta columna de precio por nombre o usa la primera numerica
candidatas_precio = [c for c in num_cols if "price" in c or "precio" in c]
col_precio = candidatas_precio[0] if candidatas_precio else num_cols[0]

otras_numericas = [c for c in num_cols if c != col_precio]
col_x = otras_numericas[0] if otras_numericas else col_precio

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_eda[col_precio], bins=30, kde=True, ax=ax, color="#2563eb")
ax.set_title(f"Histograma de {col_precio}")
plt.tight_layout()
fig.savefig(RUTA_FIGS / "01_histograma_precio.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x=df_eda[col_precio], ax=ax, color="#16a34a")
ax.set_title(f"Box plot de {col_precio}")
plt.tight_layout()
fig.savefig(RUTA_FIGS / "02_boxplot_precio.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_eda, x=col_x, y=col_precio, alpha=0.6, ax=ax, color="#9333ea")
ax.set_title(f"Dispersion: {col_precio} vs {col_x}")
plt.tight_layout()
fig.savefig(RUTA_FIGS / "03_scatter_precio_vs_variable.png", dpi=150)
plt.show()

corr = df_eda[num_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Mapa de calor de correlaciones")
plt.tight_layout()
fig.savefig(RUTA_FIGS / "04_heatmap_correlaciones.png", dpi=150)
plt.show()

print("Visualizaciones guardadas en la carpeta Visualizaciones/.")
print(f"Variable de precio usada: {col_precio}")
print(f"Variable para dispersion: {col_x}")

## 5. Interpretacion inicial de graficas

1. **Histograma de precio:** permite identificar si la distribucion del precio es simetrica o sesgada y detectar colas largas.
2. **Box plot de precio:** ayuda a detectar valores atipicos en el precio.
3. **Scatter plot:** permite ver si existe tendencia entre precio y una variable numerica relevante.
4. **Heatmap:** muestra correlaciones entre variables numericas para detectar relaciones potenciales con el precio.

Completa esta seccion con tus hallazgos concretos al ejecutar el cuaderno con el dataset final.